In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import warnings

# Setting up pandas display options for better output visualization
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# ====================================================================
# 1. PATH AND GLOBAL CONFIGURATION 🌎
# ====================================================================

print("="*80)
print("DISASTER HISTORY ANALYSIS SYSTEM (COUNTRY FOCUS) - FINAL (Period Split)")
print("="*80)

# NOTE: Adjust BASE_DIR path for your environment
BASE_DIR = Path(r"C:\Users\wb512463\OneDrive - WBG\Resilient Housing\Global analysis")
DATA_RAW = BASE_DIR / "data_raw"
DATA_PROCESSED = BASE_DIR / "data_processed"

# --- Input Data Paths ---
EMDAT_FILE = DATA_RAW / "Em-dat" / "public_emdat_custom_request_2025-11-07_f6f2cabe-ff99-4bc3-ba01-ab018fae62ec.xlsx"

# --- Output Path ---
COUNTRY_OUTPUT_CSV_FILE = DATA_PROCESSED / "country_disaster_summary_FINAL_PERIOD_SPLIT.csv"

# year definition 
YEAR_START_CURRENT = 2000
YEAR_END_CURRENT = 2020  
YEAR_START_HISTORY = 1900
YEAR_END_HISTORY = 1999

# Create output directory
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"✅ Using Target EMDAT File: {EMDAT_FILE.name}")
print(f"✅ Main Analysis Period: {YEAR_START_CURRENT}-{YEAR_END_CURRENT}")
print(f"✅ History Period: {YEAR_START_HISTORY}-{YEAR_END_HISTORY}")
print("================================================================================")

# --------------------------------------------------------------------
# Target disaster mapping 
# --------------------------------------------------------------------
DISASTER_NAME_MAP = {
    'earthquake': 'EQK', 
    'flood': 'FLD', 
    'storm': 'STM', 
    'volcanic activity': 'VOL', 
    'wildfire': 'FIRE'
}

def shorten_disaster_col(original_col_name):
    """
    Shortn EM-DAT disaster names 
    """
    lower_name = str(original_col_name).lower().strip()
    for full_name, short_code in DISASTER_NAME_MAP.items():
        if full_name == lower_name: 
            return f"DIS_{short_code}" 
    return "DROP_COL" 

EXPECTED_SHORT_COLS = sorted([f"DIS_{code}" for code in DISASTER_NAME_MAP.values()])
print(f" ✓ Target disaster column codes: {EXPECTED_SHORT_COLS}")
print("-" * 80)


# ====================================================================
# 2. LOAD AND PARSE EM-DAT DATA
# ====================================================================

print("\n[STEP 1] Loading and Parsing ALL EM-DAT Disaster Data...")

try:
    emdat_df = pd.read_excel(EMDAT_FILE, header=0)
except Exception as e:
    print(f"Error: {e}")
    exit()

COL_MAP_EXCEL_TO_INTERNAL = {
    'DisNo.': 'dis_no',
    'ISO': 'iso_code',  
    'Country': 'country_name', 
    'Disaster Type': 'disaster_type',
    'Start Year': 'year',
    'Declaration': 'declaration',
    'Total Deaths': 'total_deaths',
    'Total Affected': 'total_affected',
    'Total Damage (\'000 US$)': 'total_damage_usd' 
}

emdat_raw = emdat_df[list(COL_MAP_EXCEL_TO_INTERNAL.keys())].rename(columns=COL_MAP_EXCEL_TO_INTERNAL)
emdat_raw['dis_no'] = emdat_raw['dis_no'].astype(str)

emdat_raw['total_deaths'] = pd.to_numeric(emdat_raw['total_deaths'], errors='coerce')
emdat_raw['total_affected'] = pd.to_numeric(emdat_raw['total_affected'], errors='coerce')
emdat_raw['total_damage_usd'] = pd.to_numeric(emdat_raw['total_damage_usd'], errors='coerce')
emdat_raw['year'] = pd.to_numeric(emdat_raw['year'], errors='coerce').astype('Int64')
emdat_raw['total_damage_usd'] = emdat_raw['total_damage_usd'] * 1000
emdat_raw['disaster_type'] = emdat_raw['disaster_type'].astype(str).str.strip()


print(f"\n✅ EM-DAT raw events (all periods): {len(emdat_raw):,} 件")

emdat_raw['iso_code'] = emdat_raw['iso_code'].astype(str).str.strip()
emdat_raw['country_name'] = emdat_raw['country_name'].astype(str).str.strip()
disasters_df = emdat_raw.copy()
disasters_df.dropna(subset=['iso_code', 'year'], inplace=True)  
disasters_df = disasters_df[disasters_df['iso_code'] != '']

print(f" ✓ Usable events after minimal cleansing (ISO and year required): {len(disasters_df):,} 件")
print("-" * 80)


# ====================================================================
# 3. AGGREGATE DISASTER DATA BY COUNTRY AND PERIOD 
# ====================================================================

print("\n[STEP 2] Aggregating Disaster Counts and IMPACT by Period...")

df_current = disasters_df[
    (disasters_df['year'] >= YEAR_START_CURRENT) &
    (disasters_df['year'] <= YEAR_END_CURRENT)
].copy()

df_history = disasters_df[
    (disasters_df['year'] >= YEAR_START_HISTORY) & 
    (disasters_df['year'] <= YEAR_END_HISTORY)
].copy()

print(f" Disaster Records: 2000-2020: {len(df_current):,}  cases")
print(f" Disaster Records: 1900-1999: {len(df_history):,} cases")

GROUPING_KEY = ['iso_code', 'country_name']

# ====================================================================
# A. Main analysis(2000-2020) 
# ====================================================================

# --- 1. Disaster Events ---
total_disasters_country = df_current.groupby(GROUPING_KEY)['dis_no'].nunique().rename('DIS_TOTAL')
declared_disasters_country = df_current[df_current['declaration'].astype(str).str.upper() == 'YES'] \
    .groupby(GROUPING_KEY)['dis_no'].nunique().rename('DIS_DECLAR')

# --- 2. Evnets per disaster type ---
disaster_type_pivot_country = pd.pivot_table(
    df_current, index=GROUPING_KEY, columns='disaster_type', values='dis_no', aggfunc='nunique', fill_value=0
)
new_columns_country = {col: shorten_disaster_col(col) for col in disaster_type_pivot_country.columns}
disaster_type_pivot_country.rename(columns=new_columns_country, inplace=True)
all_disaster_cols_country = sorted(list(set(disaster_type_pivot_country.columns).union(EXPECTED_SHORT_COLS)))
disaster_type_pivot_country = disaster_type_pivot_country.reindex(columns=all_disaster_cols_country, fill_value=0)

#  --- 3. Dealth per disaster type (Robustness check) ---
disaster_deaths_pivot_current = pd.pivot_table(
    df_current, index=GROUPING_KEY, columns='disaster_type', values='total_deaths', aggfunc='sum', fill_value=0
)
new_deaths_cols_current = {
    col: f"{shorten_disaster_col(col)}_DEATHS" 
    for col in disaster_deaths_pivot_current.columns if shorten_disaster_col(col) != "DROP_COL"
}
disaster_deaths_pivot_current.rename(columns=new_deaths_cols_current, inplace=True)


# --- 4. summarize---
total_impact_country = df_current.groupby(GROUPING_KEY).agg(
    DIS_TOTAL_DEATHS=('total_deaths', 'sum'),
    DIS_TOTAL_DAMAGE_USD=('total_damage_usd', 'sum')
)

# ====================================================================
# B. Historical records  (1900-1999) - for reference 
# ====================================================================

history_impact_country = df_history.groupby(GROUPING_KEY).agg(
    DIS_TOTAL_DEATHS_History=('total_deaths', 'sum'),
    DIS_TOTAL_DAMAGE_USD_History=('total_damage_usd', 'sum')
)

#  --- 2. Death per disaster ---
disaster_deaths_pivot_history = pd.pivot_table(
    df_history, index=GROUPING_KEY, columns='disaster_type', values='total_deaths', aggfunc='sum', fill_value=0
)
new_history_deaths_cols = {
    col: f"{shorten_disaster_col(col)}_DEATHS_History" 
    for col in disaster_deaths_pivot_history.columns if shorten_disaster_col(col) != "DROP_COL"
}
disaster_deaths_pivot_history.rename(columns=new_history_deaths_cols, inplace=True)

# ====================================================================
# C. Merge all results 
# ====================================================================

country_disaster_summary = pd.concat([
    total_disasters_country, 
    declared_disasters_country.reindex(total_disasters_country.index, fill_value=0), 
    disaster_type_pivot_country,
    disaster_deaths_pivot_current, 
    total_impact_country, 
    history_impact_country,
    disaster_deaths_pivot_history 
], axis=1)

country_disaster_summary.fillna(0, inplace=True)

# Caclulate average impact per event 
impact_event_counts_current = country_disaster_summary['DIS_TOTAL']
country_disaster_summary['DIS_AVG_DEATHS'] = (
    country_disaster_summary['DIS_TOTAL_DEATHS'] / impact_event_counts_current
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

country_disaster_summary['DIS_AVG_DAMAGE_USD'] = (
    country_disaster_summary['DIS_TOTAL_DAMAGE_USD'] / impact_event_counts_current
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

new_death_cols_current_list = list(new_deaths_cols_current.values())
new_death_cols_history_list = list(new_history_deaths_cols.values())

int_cols = (
    ['DIS_TOTAL', 'DIS_DECLAR'] + 
    all_disaster_cols_country + 
    new_death_cols_current_list + 
    ['DIS_TOTAL_DEATHS', 'DIS_TOTAL_DAMAGE_USD'] + 
    ['DIS_TOTAL_DEATHS_History', 'DIS_TOTAL_DAMAGE_USD_History'] +
    new_death_cols_history_list 
)

for col in int_cols:
    if col in country_disaster_summary.columns:
        country_disaster_summary[col] = country_disaster_summary[col].astype(int)

country_disaster_summary.reset_index(inplace=True)

print(f"\n ✓ Aggregated disaster data for {len(country_disaster_summary):,} unique countries.")
print(f" ✓ Total columns in country summary: {len(country_disaster_summary.columns)}")
print("-" * 80)


# ====================================================================
# 4. SAVE FINAL DATA AND SUMMARY STATISTICS 💾 
# ====================================================================

print("\n[STEP 3] Saving Final Data and Statistics...")

# 1. National summary
country_disaster_summary.to_csv(COUNTRY_OUTPUT_CSV_FILE, index=False, encoding='utf-8-sig')
print(f" ✓ Successfully saved Country-Level data to: {COUNTRY_OUTPUT_CSV_FILE}")

# 2. Summary stats
summary_stats = {
    'Total EMDAT Events (Raw)': len(emdat_df),
    'Usable Events (ISO and Year present)': len(disasters_df),
    'Events (2000-2020)': len(df_current),
    'Events (1900-1999)': len(df_history),
    'Number of Countries with Disasters (2000-2023)': country_disaster_summary[country_disaster_summary['DIS_TOTAL'] > 0]['iso_code'].nunique(), 
}

# ====================================================================
# 5. DISPLAY FINAL SUMMARY 
# ====================================================================

print("\n" + "="*80)
print("FINAL STATISTICAL SUMMARY")
print("="*80)
for key, value in summary_stats.items():
    if isinstance(value, float):
        print(f" {key}: {value:,.2f}")
    else:
        print(f" {key}: {value:,}")
print("="*80)
print("Processing complete.")

DISASTER HISTORY ANALYSIS SYSTEM (COUNTRY FOCUS) - FINAL (Period Split)
✅ Using Target EMDAT File: public_emdat_custom_request_2025-11-07_f6f2cabe-ff99-4bc3-ba01-ab018fae62ec.xlsx
✅ Main Analysis Period: 2000-2020
✅ History Period: 1900-1999
 ✓ Target disaster column codes: ['DIS_EQK', 'DIS_FIRE', 'DIS_FLD', 'DIS_STM', 'DIS_VOL']
--------------------------------------------------------------------------------

[STEP 1] Loading and Parsing ALL EM-DAT Disaster Data...

✅ EM-DAT raw events (all periods): 12,986 件
 ✓ Usable events after minimal cleansing (ISO and year required): 12,986 件
--------------------------------------------------------------------------------

[STEP 2] Aggregating Disaster Counts and IMPACT by Period...
 Disaster Records: 2000-2020: 6,480  cases
 Disaster Records: 1900-1999: 5,417 cases

 ✓ Aggregated disaster data for 223 unique countries.
 ✓ Total columns in country summary: 25
--------------------------------------------------------------------------------

[STE